# 第6章  矩阵即变换 —— 从表格到空间扭曲

> 矩阵乘法的代数(行x列点积)和几何(空间扭曲)视角。手撕全连接层。广播机制+连续批处理。
> 第5章(点积)、第4章(向量)、第3章(shape)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print('环境就绪')


---
## 6.1-6.3  矩阵乘法+几何变换

W@x=每行与x点积。A@B:三重循环vs@运算符(BLAS加速100-10000x)。网格旋转/拉伸/剪切可视化。

AI连接：第29章Q@K^T就是批量矩阵乘法。


In [ ]:
import numpy as np
W=np.array([[1.,2.],[3.,4.],[5.,6.]]); x=np.array([2.,1.])
print('W@x=',W@x)
# Rotation preserves length
th=np.pi/4; R=np.array([[np.cos(th),-np.sin(th)],[np.sin(th),np.cos(th)]])
v=np.array([3.,4.]); print(f'|v|={np.linalg.norm(v):.4f} |R@v|={np.linalg.norm(R@v):.4f}')


---
## 6.4-6.5  全连接层+广播机制

output=X@W.T+b。广播:从右往左对齐,相等或为1兼容,缺失补1。

AI连接：第28章两层网络=两次FC+ReLU。


In [ ]:
import numpy as np
X=np.random.randn(32,784); W=np.random.randn(256,784)*.01; b=np.zeros(256)
out=X@W.T+b; print('FC:',X.shape,'@',W.T.shape,'+',b.shape,'->',out.shape)
a=np.ones((3,1)); b2=np.ones((1,4)); print('(3,1)+(1,4)->',(a+b2).shape)
try: np.ones((3,4))+np.ones((3,2))
except ValueError: print('(3,4)+(3,2)->ValueError(incompatible)')


---
## 6.6  Agent连续批处理 🆕

变长序列拼接->(total_tokens,d)批量矩阵->一次matmul。块对角Attention Mask隔离用户。

AI连接：vLLM/TensorRT-LLM工业落地。


In [ ]:
import numpy as np
np.random.seed(42)
seq_lens=[3,5,2,4]; d=8
embs=[np.random.randn(l,d) for l in seq_lens]
X=np.vstack(embs); total=sum(seq_lens)
print('total tokens:',total,'vs padded:',4*max(seq_lens))
mask=np.zeros((total,total)); off=0
for l in seq_lens: mask[off:off+l,off:off+l]=1; off+=l
print('Block-diag mask (0-2=user0,3-7=user1,...):')
print(mask.astype(int))
WQ=np.random.randn(d,d)*.1; Q=X@WQ; K=X@WQ
s=Q@K.T/np.sqrt(d); sm=np.where(mask==1,s,-np.inf)
print('Cross-user blocked:',sm[0,3]==-np.inf)


---
## 习题

> 在下方新建代码单元格作答。

1.(概念)A@B的前提条件？C[i,j]怎么算？
2.(概念)(3,4)@(4,5)可以但(3,4)@(3,4)不行？
3.(概念)广播三条规则？(32,1,10)+(1,5,10)的shape？
4.(代码)旋转矩阵A@x验证长度不变。
5.(概念)连续批处理mask为什么块对角？全设1后果？
6.(代码)50学生×4科广播标准化+加权求和。


---

> 章末钩子：矩阵乘法批量做点积。有些矩阵可逆向操作——逆矩阵。
预览：下一章——**矩阵的逆与线性方程组**。

> 提示：完成后运行 Kernel -> Restart & Run All 验证。
